In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [2]:
#remove 2 suspecious outliers
train = train[~((train['GrLivArea'] > 4000) & 
                (train['SalePrice'] < 200000))]
print('Train shape after removing outliers:', train.shape)

Train shape after removing outliers: (1458, 81)


In [3]:
#separate target variable befor transformation

y_train = np.log1p(train['SalePrice'])

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop(['Id'], axis=1)

print('Train features shape:', train.shape)
print('Test shape:', test.shape)
print('Target shape:', y_train.shape)
print('\nFirst 5 target values (log transformed):')
print(y_train.head())

Train features shape: (1458, 79)
Test shape: (1459, 79)
Target shape: (1458,)

First 5 target values (log transformed):
0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [4]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 
             'FireplaceQu', 'GarageType', 'GarageFinish',
             'GarageQual', 'GarageCond', 'BsmtQual', 
             'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
             'BsmtFinType2', 'MasVnrType']

for col in none_cols:
    train[col] = train[col].fillna('None')
    test[col]  = test[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageCars', 'GarageArea',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'TotalBsmtSF', 'MasVnrArea', 'BsmtFullBath', 
             'BsmtHalfBath']

for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

lot_medians = train.groupby('Neighborhood')['LotFrontage'].median()
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
test['LotFrontage'] = test['Neighborhood'].map(lot_medians).where(test['LotFrontage'].isna(), test['LotFrontage'])

In [5]:
# Electrical → 1 missing, fill with mode from TRAIN
electrical_mode = train['Electrical'].mode()[0]
train['Electrical'] = train['Electrical'].fillna(electrical_mode)
test['Electrical']  = test['Electrical'].fillna(electrical_mode)

# Row 332 and 948 specific fixes 
bsmt_fintype2_mode = train['BsmtFinType2'].mode()[0]
bsmt_exposure_mode = train['BsmtExposure'].mode()[0]

train['BsmtFinType2'] = train['BsmtFinType2'].fillna(bsmt_fintype2_mode)
test['BsmtFinType2']  = test['BsmtFinType2'].fillna(bsmt_fintype2_mode)

train['BsmtExposure'] = train['BsmtExposure'].fillna(bsmt_exposure_mode)
test['BsmtExposure']  = test['BsmtExposure'].fillna(bsmt_exposure_mode)

In [6]:
# Verify: no missing values remaining
train_missing = train.isnull().sum().sum()
test_missing  = test.isnull().sum().sum()

print(f'Train missing values remaining: {train_missing}')
print(f'Test missing values remaining:  {test_missing}')

Train missing values remaining: 0
Test missing values remaining:  12


In [7]:
# find which column still has missing value in train
print("Train missing columns:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest missing columns:")
print(test.isnull().sum()[test.isnull().sum() > 0])

Train missing columns:
Series([], dtype: int64)

Test missing columns:
MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64


In [8]:
# columns that need mode filling
mode_cols = ['Electrical', 'MSZoning', 'Utilities', 
             'Exterior1st', 'Exterior2nd', 'KitchenQual', 
             'Functional', 'SaleType']

for col in mode_cols:
    mode_val = train[col].mode()[0]
    
    train[col] = train[col].fillna(mode_val)
    test[col]  = test[col].fillna(mode_val)

# verify
print(f'Train missing: {train.isnull().sum().sum()}')
print(f'Test missing:  {test.isnull().sum().sum()}')

Train missing: 0
Test missing:  0


In [9]:
# CREATE NEW FEATURES 

# total area of entire house
train['TotalSF'] = (train['TotalBsmtSF'] + 
                    train['1stFlrSF'] + 
                    train['2ndFlrSF'])
test['TotalSF']  = (test['TotalBsmtSF'] + 
                    test['1stFlrSF'] + 
                    test['2ndFlrSF'])

# total bathrooms (half bath counts as 0.5)
train['TotalBaths'] = (train['FullBath'] + 
                       train['BsmtFullBath'] +
                       train['HalfBath'] * 0.5 + 
                       train['BsmtHalfBath'] * 0.5)
test['TotalBaths']  = (test['FullBath'] + 
                       test['BsmtFullBath'] +
                       test['HalfBath'] * 0.5 + 
                       test['BsmtHalfBath'] * 0.5)

# age features — how old when sold
train['HouseAge']      = train['YrSold'] - train['YearBuilt']
train['RenovationAge'] = train['YrSold'] - train['YearRemodAdd']
train['IsRenovated']   = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)

test['HouseAge']       = test['YrSold'] - test['YearBuilt']
test['RenovationAge']  = test['YrSold'] - test['YearRemodAdd']
test['IsRenovated']    = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

# porch total area
train['TotalPorch'] = (train['OpenPorchSF'] + 
                       train['EnclosedPorch'] +
                       train['3SsnPorch'] + 
                       train['ScreenPorch'])
test['TotalPorch']  = (test['OpenPorchSF'] + 
                       test['EnclosedPorch'] +
                       test['3SsnPorch'] + 
                       test['ScreenPorch'])

# binary features — has or doesn't have
train['HasPool']      = (train['PoolArea'] > 0).astype(int)
train['HasGarage']    = (train['GarageArea'] > 0).astype(int)
train['HasBasement']  = (train['TotalBsmtSF'] > 0).astype(int)
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)

test['HasPool']       = (test['PoolArea'] > 0).astype(int)
test['HasGarage']     = (test['GarageArea'] > 0).astype(int)
test['HasBasement']   = (test['TotalBsmtSF'] > 0).astype(int)
test['HasFireplace']  = (test['Fireplaces'] > 0).astype(int)

# quality * condition interaction
train['OverallScore'] = train['OverallQual'] * train['OverallCond']
test['OverallScore']  = test['OverallQual'] * test['OverallCond']

# verify new features added
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('\nNew features created:')
new_features = ['TotalSF', 'TotalBaths', 'HouseAge', 'RenovationAge',
                'IsRenovated', 'TotalPorch', 'HasPool', 'HasGarage',
                'HasBasement', 'HasFireplace', 'OverallScore']
print(train[new_features].head())

Train shape: (1458, 90)
Test shape: (1459, 90)

New features created:
   TotalSF  TotalBaths  HouseAge  RenovationAge  IsRenovated  TotalPorch  \
0     2566         3.5         5              5            0          61   
1     2524         2.5        31             31            0           0   
2     2706         3.5         7              6            1          42   
3     2473         2.0        91             36            1         307   
4     3343         3.5         8              8            0          84   

   HasPool  HasGarage  HasBasement  HasFireplace  OverallScore  
0        0          1            1             0            35  
1        0          1            1             1            48  
2        0          1            1             1            35  
3        0          1            1             1            35  
4        0          1            1             1            40  


In [10]:
drop_cols = ['1stFlrSF', 'YearBuilt', 'YearRemodAdd']
train = train.drop(drop_cols, axis=1)
test  = test.drop(drop_cols, axis=1)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

# verify dropped columns are gone
dropped = ['1stFlrSF', 'YearBuilt', 'YearRemodAdd']
for col in dropped:
    print(f'{col} in train: {col in train.columns}')

Train shape: (1458, 87)
Test shape: (1459, 87)
1stFlrSF in train: False
YearBuilt in train: False
YearRemodAdd in train: False


In [11]:
nums_cols = train.select_dtypes(include=['int64', 'float64']).columns

skewness = train[nums_cols].skew().sort_values(ascending=False)

print("Highly skewed features (skewness > 0.5):")
print(skewness[skewness > 0.5])

Highly skewed features (skewness > 0.5):
MiscVal          24.460085
PoolArea         15.948945
HasPool          15.508026
LotArea          12.573925
3SsnPorch        10.297106
LowQualFinSF      9.004955
KitchenAbvGr      4.484883
BsmtFinSF2        4.251925
ScreenPorch       4.118929
BsmtHalfBath      4.100114
EnclosedPorch     3.087164
MasVnrArea        2.696329
OpenPorchSF       2.339829
TotalPorch        2.011471
LotFrontage       1.548217
WoodDeckSF        1.545805
MSSubClass        1.407011
GrLivArea         1.010992
BsmtUnfSF         0.920903
2ndFlrSF          0.812957
TotalSF           0.804321
BsmtFinSF1        0.764789
OverallCond       0.691035
HalfBath          0.680051
TotRmsAbvGrd      0.660502
Fireplaces        0.632060
HouseAge          0.607894
BsmtFullBath      0.590358
TotalBsmtSF       0.511703
RenovationAge     0.500821
dtype: float64


In [12]:
from scipy import stats

# columns to EXCLUDE from skewness correction
exclude_cols = ['HasPool', 'HasGarage', 'HasBasement', 
                'HasFireplace', 'IsRenovated',
                'OverallQual', 'OverallCond', 'OverallScore']

# get numerical columns excluding binary/ordinal
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
num_cols = [col for col in num_cols if col not in exclude_cols]

# calculate skewness
skewness = train[num_cols].skew()

# get columns with skewness > 0.5
skewed_cols = skewness[skewness > 0.5].index.tolist()
print(f'Columns to fix: {len(skewed_cols)}')
print(skewed_cols)

Columns to fix: 28
['MSSubClass', 'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'HalfBath', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'TotalSF', 'HouseAge', 'RenovationAge', 'TotalPorch']


In [13]:
skewness_before = train[skewed_cols].skew()

# apply log1p to all skewed columns
for col in skewed_cols:
    train[col] = np.log1p(train[col])
    test[col]  = np.log1p(test[col])

# verify skewness is reduced
skewness_after = train[skewed_cols].skew()

print("Skewness BEFORE vs AFTER:")
comparison = pd.DataFrame({
    'before': skewness_before,
    'after' : skewness_after
}).round(2)
print(comparison)
print(f'\nColumns still skewed above 0.5: {(skewness_after > 0.5).sum()}')

Skewness BEFORE vs AFTER:
               before  after
MSSubClass       1.41   0.25
LotFrontage      1.55  -1.00
LotArea         12.57  -0.18
MasVnrArea       2.70   0.51
BsmtFinSF1       0.76  -0.62
BsmtFinSF2       4.25   2.52
BsmtUnfSF        0.92  -2.18
TotalBsmtSF      0.51  -5.17
2ndFlrSF         0.81   0.29
LowQualFinSF     9.00   7.45
GrLivArea        1.01  -0.07
BsmtFullBath     0.59   0.42
BsmtHalfBath     4.10   3.93
HalfBath         0.68   0.57
KitchenAbvGr     4.48   3.87
TotRmsAbvGrd     0.66  -0.07
Fireplaces       0.63   0.18
WoodDeckSF       1.55   0.16
OpenPorchSF      2.34  -0.02
EnclosedPorch    3.09   2.11
3SsnPorch       10.30   7.73
ScreenPorch      4.12   3.15
PoolArea        15.95  15.52
MiscVal         24.46   5.17
TotalSF          0.80  -0.47
HouseAge         0.61  -0.84
RenovationAge    0.50  -0.57
TotalPorch       2.01  -0.51

Columns still skewed above 0.5: 11


/home/anurag/house-price-prediction/venv/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/anurag/house-price-prediction/venv/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [14]:
# check for negative values in skewed columns
print("Columns with negative values:")
for col in skewed_cols:
    neg_count = (train[col] < 0).sum()
    if neg_count > 0:
        print(f'{col}: {neg_count} negative values')
        print(f'  min value: {train[col].min():.3f}')

Columns with negative values:


In [15]:
print("Test columns with negative values:")
for col in skewed_cols:
    neg_count = (test[col] < 0).sum()
    if neg_count > 0:
        print(f'{col}: {neg_count} negative values')
        print(f'  min value: {test[col].min():.3f}')

Test columns with negative values:
HouseAge: 1 negative values
  min value: -inf
RenovationAge: 1 negative values
  min value: -inf


In [16]:
# check for NaN values in skewed columns
# NaN can also cause this warning
print("Columns with NaN after transformation:")
for col in skewed_cols:
    nan_train = train[col].isna().sum()
    nan_test  = test[col].isna().sum()
    if nan_train > 0 or nan_test > 0:
        print(f'{col}: train={nan_train}, test={nan_test}')

Columns with NaN after transformation:
RenovationAge: train=0, test=1


In [17]:
# fill NaN in age features using train median
for col in ['HouseAge', 'RenovationAge']:
    median_val = train[col].median()
    test[col]  = test[col].fillna(median_val)
    print(f'{col} median fill value: {median_val:.3f}')

# verify
print(f'\nNaN remaining in HouseAge:     {test["HouseAge"].isna().sum()}')
print(f'NaN remaining in RenovationAge: {test["RenovationAge"].isna().sum()}')

HouseAge median fill value: 3.584
RenovationAge median fill value: 2.708

NaN remaining in HouseAge:     0
NaN remaining in RenovationAge: 0


In [18]:
# drop sparse columns that are already captured
# by our engineered features
sparse_cols = ['PoolArea', '3SsnPorch', 'ScreenPorch', 
               'EnclosedPorch', 'MiscVal', 'BsmtHalfBath',
               'BsmtFinSF2']

train = train.drop(sparse_cols, axis=1)
test  = test.drop(sparse_cols, axis=1)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (1458, 80)
Test shape: (1459, 80)


In [19]:
# ordinal mappings — order matters here!
# we define LOW to HIGH for each column

ordinal_mappings = {
    'ExterQual'  : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond'  : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual'   : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond'   : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC'  : ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual' : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond' : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC'     : ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence'      : ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'LotShape'   : ['IR3', 'IR2', 'IR1', 'Reg'],
    'LandSlope'  : ['Sev', 'Mod', 'Gtl'],
    'PavedDrive' : ['N', 'P', 'Y'],
    'Utilities'  : ['NoSeWa', 'AllPub'],
    'Functional' : ['Sal', 'Sev', 'Maj2', 'Maj1', 
                    'Mod', 'Min2', 'Min1', 'Typ'],
}

# apply ordinal encoding
for col, order in ordinal_mappings.items():

    mapping = {val: idx for idx, val in enumerate(order)}
    
    train[col] = train[col].map(mapping)
    test[col]  = test[col].map(mapping)

print('Ordinal encoding done!')
print(train[list(ordinal_mappings.keys())].head(3))

Ordinal encoding done!
   ExterQual  ExterCond  BsmtQual  BsmtCond  BsmtExposure  BsmtFinType1  \
0          3          2         4         3             1             6   
1          2          2         4         3             4             5   
2          3          2         4         3             2             6   

   BsmtFinType2  HeatingQC  KitchenQual  FireplaceQu  GarageFinish  \
0             1          4            3            0             2   
1             1          4            2            3             2   
2             1          4            3            3             2   

   GarageQual  GarageCond  PoolQC  Fence  LotShape  LandSlope  PavedDrive  \
0           3           3       0      0         3          2           2   
1           3           3       0      0         3          2           2   
2           3           3       0      0         2          2           2   

   Utilities  Functional  
0          1           7  
1          1           7  
2    

In [20]:
# CentralAir: Y→1, N→0
train['CentralAir'] = train['CentralAir'].map({'Y': 1, 'N': 0})
test['CentralAir']  = test['CentralAir'].map({'Y': 1, 'N': 0})

print(train['CentralAir'].value_counts())

CentralAir
1    1363
0      95
Name: count, dtype: int64


In [21]:
# find remaining categorical columns
remaining_cat = train.select_dtypes(include=['object']).columns.tolist()
print(f'Remaining categorical columns: {len(remaining_cat)}')
print(remaining_cat)

Remaining categorical columns: 22
['MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical', 'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition']


In [22]:
# check unique values per remaining categorical column
for col in remaining_cat:
    n_unique = train[col].nunique()
    print(f'{col:20} → {n_unique} unique values')

MSZoning             → 5 unique values
Street               → 2 unique values
Alley                → 3 unique values
LandContour          → 4 unique values
LotConfig            → 5 unique values
Neighborhood         → 25 unique values
Condition1           → 9 unique values
Condition2           → 8 unique values
BldgType             → 5 unique values
HouseStyle           → 8 unique values
RoofStyle            → 6 unique values
RoofMatl             → 7 unique values
Exterior1st          → 15 unique values
Exterior2nd          → 16 unique values
MasVnrType           → 4 unique values
Foundation           → 6 unique values
Heating              → 6 unique values
Electrical           → 5 unique values
GarageType           → 7 unique values
MiscFeature          → 5 unique values
SaleType             → 9 unique values
SaleCondition        → 6 unique values


In [23]:
# check how dominant 'None' is in sparse columns
sparse_check = ['Alley', 'MiscFeature', 'Street']
for col in sparse_check:
    value_counts = train[col].value_counts(normalize=True) * 100
    print(f'\n{col}:')
    print(value_counts.round(1))


Alley:
Alley
None    93.8
Grvl     3.4
Pave     2.8
Name: proportion, dtype: float64

MiscFeature:
MiscFeature
None    96.3
Shed     3.4
Gar2     0.1
Othr     0.1
TenC     0.1
Name: proportion, dtype: float64

Street:
Street
Pave    99.6
Grvl     0.4
Name: proportion, dtype: float64


In [24]:
#drop near-constant columns
drop_sparse = ['Alley', 'MiscFeature', 'Street']
train = train.drop(drop_sparse, axis=1)
test  = test.drop(drop_sparse, axis=1)

print('After dropping sparse:', train.shape)

After dropping sparse: (1458, 77)


In [25]:
# simple mean encoding for Neighborhood for now
neighborhood_means = y_train.groupby(train['Neighborhood']).mean()

train['Neighborhood'] = train['Neighborhood'].map(neighborhood_means)
test['Neighborhood']  = test['Neighborhood'].map(neighborhood_means)

# check remaining categorical columns
remaining_cat = train.select_dtypes(include=['object']).columns.tolist()
print(f'Columns to one hot encode: {len(remaining_cat)}')
print(remaining_cat)

Columns to one hot encode: 18
['MSZoning', 'LandContour', 'LotConfig', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical', 'GarageType', 'SaleType', 'SaleCondition']


In [29]:
# one hot encode remaining categorical columns
# drop_first=True removes one redundant column per feature
# we apply to train and test together to ensure
# same columns are created in both

# store original lengths to split later
train_len = len(train)

# combine for encoding
combined = pd.concat([train, test], axis=0)

# one hot encode
combined = pd.get_dummies(combined, 
                          columns=remaining_cat,
                          drop_first=True)

# split back
train = combined[:train_len].copy()
test  = combined[train_len:].copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)

KeyError: "None of [Index(['MSZoning', 'LandContour', 'LotConfig', 'Condition1', 'Condition2',\n       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',\n       'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical',\n       'GarageType', 'SaleType', 'SaleCondition'],\n      dtype='object')] are in the [columns]"

In [27]:
# verify no missing values
print(f'Train missing: {train.isnull().sum().sum()}')
print(f'Test missing:  {test.isnull().sum().sum()}')

# verify dtypes — should be all numeric now
print(f'\nTrain dtypes:')
print(train.dtypes.value_counts())

# verify shapes match
print(f'\nTrain shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Target shape: {y_train.shape}')

Train missing: 0
Test missing:  0

Train dtypes:
bool       113
int64       33
float64     26
Name: count, dtype: int64

Train shape: (1458, 172)
Test shape:  (1459, 172)
Target shape: (1458,)


In [30]:
# convert bool columns to int
bool_cols = train.select_dtypes(include=['bool']).columns
train[bool_cols] = train[bool_cols].astype(int)
test[bool_cols]  = test[bool_cols].astype(int)

# verify
print('Train dtypes after fix:')
print(train.dtypes.value_counts())

Train dtypes after fix:
int64      146
float64     26
Name: count, dtype: int64


In [31]:
# save processed data
train.to_csv('../data/train_processed.csv', index=False)
test.to_csv('../data/processed_test.csv', index=False)
np.save('../data/y_train.npy', y_train.values)

print('Data saved successfully!')
print('Files created:')
print('  data/train_processed.csv')
print('  data/processed_test.csv')
print('  data/y_train.npy')

Data saved successfully!
Files created:
  data/train_processed.csv
  data/processed_test.csv
  data/y_train.npy
